<div style="background:#0f3460;padding:40px;border-radius:12px;text-align:center;">

<h1 style="color:#e94560;font-size:28px;font-weight:bold;margin:0 0 12px 0;">
  Taller modelación # 2
</h1>

**Integrantes:**

* Valentina Giraldo Gaviria
* Héctor Hernan Betancourt López
* Marcela Fajardo Bermúdez
* Rafael Chamorro

In [ ]:
# Importamos PuLP
import pulp as lp

#Ejercicio 1

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1. Conjuntos
# ─────────────────────────────────────────────────────────────

nombres_series = ["House of the Dragon", "Peaky Blinders",
                  "Homeland", "Dahmer", "Breaking Bad",
                  "Stranger Things", "Dexter"]

series = [0, 1, 2, 3, 4, 5, 6]
# 0=House of the Dragon, 1=Peaky Blinders, 2=Homeland,
# 3=Dahmer, 4=Breaking Bad, 5=Stranger Things, 6=Dexter

# ─────────────────────────────────────────────────────────────
# 2. Parámetros
# ─────────────────────────────────────────────────────────────

# Ci: costo de compra de la serie i ($)
Ci = [30000, 35000, 20000, 25000, 20000, 15000, 40000]

# COi: costo de oportunidad por hora de la serie i ($/hora)
COi = [1200, 7000, 8000, 10000, 9000, 7000, 1000]

# D: total de horas disponibles para ser asignadas
D = 20

# HX: horas máximas por serie
HX = 8

# ─────────────────────────────────────────────────────────────
# 3. Modelo
# ─────────────────────────────────────────────────────────────

modelo_series = LpProblem("Series_TV", sense=LpMinimize)

# ─────────────────────────────────────────────────────────────
# 4. Variables de decisión
# ─────────────────────────────────────────────────────────────

# Yi: 1 si compra la serie i, 0 lo contrario (binaria)
Y = LpVariable.dicts("Y", series, cat=LpBinary)

# Xi: horas semanales invertidas viendo la serie i (continua >= 0)
X = LpVariable.dicts("X", series, lowBound=0, cat=LpContinuous)

# ─────────────────────────────────────────────────────────────
# 5. Función objetivo
# MIN Z = SUM(Ci * Yi) + SUM(COi * Xi)
# Costo total de compra + Costo de oportunidad por horas vistas
# ─────────────────────────────────────────────────────────────

modelo_series += (
    lpSum(Ci[i]  * Y[i] for i in series) +
    lpSum(COi[i] * X[i] for i in series)
), "Minimizar_Costo_Total"

# ─────────────────────────────────────────────────────────────
# 6. Restricciones
# ─────────────────────────────────────────────────────────────

# R1: Horas totales disponibles para esparcimiento
# SUM(Xi) = D
# El amigo DEBE asignar las 20 horas ("invertir bien su tiempo")

modelo_series += (
    lpSum(X[i] for i in series) == D,
    "R1_Horas_totales_asignadas"
)

# R2: Límite de horas por serie i que se compra
# Xi <= HX * Yi  para todo i
# Si Yi=0 → Xi <= 0 (no puede ver si no compró)
# Si Yi=1 → Xi <= 8 (puede ver hasta HX horas)

for i in series:
    modelo_series += (
        X[i] <= HX * Y[i],
        f"R2_Limite_horas_{nombres_series[i].replace(' ','_')}"
    )

# R3: Dahmer (i=3) y Stranger Things (i=5) no pueden verse juntas
# Y4 + Y6 <= 1

modelo_series += (
    Y[3] + Y[5] <= 1,
    "R3_Dahmer_y_StrangerThings_no_juntas"
)

# R4: No negatividad  → declarada en lowBound=0 de X
# R5: Binariedad      → declarada en cat=LpBinary de Y

# ─────────────────────────────────────────────────────────────
# 7. Resolver
# ─────────────────────────────────────────────────────────────

modelo_series.solve(PULP_CBC_CMD(msg=0))

# ─────────────────────────────────────────────────────────────
# 8. Resultados
# ─────────────────────────────────────────────────────────────

print("=" * 60)
print("CASO: SERIES DE TV")
print("=" * 60)
print(f"Estado              : {LpStatus[modelo_series.status]}")
print(f"Costo mínimo Z*     : ${value(modelo_series.objective):,.0f}")
print()

# Series seleccionadas
print("Series seleccionadas:")
for i in series:
    if Y[i].value() == 1:
        print(f"  {nombres_series[i]:22}: "
              f"{X[i].value():.1f} horas  |  "
              f"Compra: ${Ci[i]:,}  |  "
              f"Oport.: ${COi[i]*X[i].value():,.0f}")

# Totales
print()
print("Desglose del costo:")
costo_compra = sum(Ci[i]  * Y[i].value() for i in series)
costo_opor   = sum(COi[i] * X[i].value() for i in series)
horas_usadas = sum(X[i].value() for i in series)
print(f"  Costo de compra      : ${costo_compra:>10,.0f}")
print(f"  Costo de oportunidad : ${costo_opor:>10,.0f}")
print(f"  COSTO TOTAL Z*       : ${value(modelo_series.objective):>10,.0f}")
print(f"  Horas asignadas      : {horas_usadas:.0f} / {D}")

# Verificación de restricciones
print()
print("Verificación de restricciones:")
print(f"  R1 Horas totales : {horas_usadas:.0f} = {D}  "
      f"{'✅' if abs(horas_usadas - D) < 0.01 else '❌'}")
for i in series:
    if Y[i].value() == 1:
        xi = X[i].value()
        yi = Y[i].value()
        print(f"  R2 {nombres_series[i]:22}: "
              f"{xi:.1f} <= {HX}*{yi:.0f} = {HX*yi:.0f}  "
              f"{'✅' if xi <= HX*yi + 0.001 else '❌'}")
d  = Y[3].value()
st = Y[5].value()
print(f"  R3 Dahmer+ST     : {d:.0f}+{st:.0f} = {d+st:.0f} <= 1  "
      f"{'✅' if d+st <= 1 else '❌'}")
print("=" * 60)

CASO: SERIES DE TV
Estado              : Optimal
Costo mínimo Z*     : $130,600

Series seleccionadas:
  House of the Dragon   : 8.0 horas  |  Compra: $30,000  |  Oport.: $9,600
  Stranger Things       : 4.0 horas  |  Compra: $15,000  |  Oport.: $28,000
  Dexter                : 8.0 horas  |  Compra: $40,000  |  Oport.: $8,000

Desglose del costo:
  Costo de compra      : $    85,000
  Costo de oportunidad : $    45,600
  COSTO TOTAL Z*       : $   130,600
  Horas asignadas      : 20 / 20

Verificación de restricciones:
  R1 Horas totales : 20 = 20  ✅
  R2 House of the Dragon   : 8.0 <= 8*1 = 8  ✅
  R2 Stranger Things       : 4.0 <= 8*1 = 8  ✅
  R2 Dexter                : 8.0 <= 8*1 = 8  ✅
  R3 Dahmer+ST     : 0+1 = 1 <= 1  ✅


#Ejercicio 2

In [ ]:
# ============================================================
# 1. Conjuntos
# ============================================================

lanzadores = ["RS", "BS", "DE", "ST", "TS"]

nombres_lanzadores = {
    "RS": "Rick Sutcliffe",
    "BS": "Bruce Sutter",
    "DE": "Dennis Eckersley",
    "ST": "Steve Trout",
    "TS": "Tim Stoddard"
}

# ============================================================
# 2. Parámetros
# ============================================================

# Costo de contratación en millones de dólares
costo = {
    "RS": 6,
    "BS": 4,
    "DE": 3,
    "ST": 2,
    "TS": 2
}

# Victorias que agrega cada lanzador
victorias = {
    "RS": 6,
    "BS": 5,
    "DE": 3,
    "ST": 3,
    "TS": 2
}

# Lanzadores derechos
# RS, BS, DE y TS son derechos
# ST es zurdo
derecho = {
    "RS": 1,
    "BS": 1,
    "DE": 1,
    "ST": 0,
    "TS": 1
}

presupuesto = 12
max_derechos = 2

# ============================================================
# 3. Modelo
# ============================================================

modelo_cubs = lp.LpProblem("Modelo_Cubs", sense=lp.LpMaximize)

# ============================================================
# 4. Variables de decisión
# ============================================================

# x[i] = 1 si el lanzador i es contratado
# x[i] = 0 si el lanzador i no es contratado

x = lp.LpVariable.dicts(
    "contratar",
    lanzadores,
    lowBound=0,
    upBound=1,
    cat=lp.LpBinary
)

# ============================================================
# 5. Función objetivo
# ============================================================

# Maximizar el total de victorias agregadas al equipo

modelo_cubs += lp.lpSum(victorias[i] * x[i] for i in lanzadores)

# ============================================================
# 6. Restricción de presupuesto
# ============================================================

# Como máximo se pueden gastar 12 millones de dólares

modelo_cubs += lp.lpSum(costo[i] * x[i] for i in lanzadores) <= presupuesto

# ============================================================
# 7. Restricción lógica
# ============================================================

# Si DE y ST son contratados, entonces BS no puede ser contratado.
# e modela como:
# x_DE + x_ST + x_BS <= 2

modelo_cubs += x["DE"] + x["ST"] + x["BS"] <= 2

# ============================================================
# 8. Restricción de lanzadores derechos
# ============================================================

# A lo sumo se pueden contratar dos lanzadores derechos

modelo_cubs += lp.lpSum(derecho[i] * x[i] for i in lanzadores) <= max_derechos

# ============================================================
# 9. Restricción de incompatibilidad entre BS y RS
# ============================================================

# Los Cubs no pueden contratar simultáneamente a BS y RS

modelo_cubs += x["BS"] + x["RS"] <= 1

# ============================================================
# 10. Resolver
# ============================================================

modelo_cubs.solve()

# ============================================================
# 11. Resultados
# ============================================================

print("=" * 60)
print("EJERCICIO 2 - Cubs")
print("=" * 60)

print("Estado:", lp.LpStatus[modelo_cubs.status])
print("Máximo número de victorias agregadas:", lp.value(modelo_cubs.objective))

print("\nLanzadores contratados:")

for i in lanzadores:
    if x[i].value() == 1:
        print(f"{i} - {nombres_lanzadores[i]}")

# ============================================================
# 12. Verificación de restricciones
# ============================================================

costo_total = sum(costo[i] * x[i].value() for i in lanzadores)
victorias_totales = sum(victorias[i] * x[i].value() for i in lanzadores)
total_derechos = sum(derecho[i] * x[i].value() for i in lanzadores)

print("\nVerificación del modelo:")
print("Costo total en millones:", costo_total)
print("Presupuesto máximo:", presupuesto)
print("Victorias totales agregadas:", victorias_totales)
print("Número de lanzadores derechos:", total_derechos)

print("\nValores de las variables:")
for i in lanzadores:
    print(f"x_{i} = {int(x[i].value())}")

EJERCICIO 2 - Cubs
Estado: Optimal
Máximo número de victorias agregadas: 12.0

Lanzadores contratados:
RS - Rick Sutcliffe
DE - Dennis Eckersley
ST - Steve Trout

Verificación del modelo:
Costo total en millones: 11.0
Presupuesto máximo: 12
Victorias totales agregadas: 12.0
Número de lanzadores derechos: 2.0

Valores de las variables:
x_RS = 1
x_BS = 0
x_DE = 1
x_ST = 1
x_TS = 0


#Ejercicio 3

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1. Conjuntos
# ─────────────────────────────────────────────────────────────
C = range(1, 9)       # canciones 1,...,8
P = range(1, 3)       # partes 1,2

# ─────────────────────────────────────────────────────────────
# Parámetros
# ─────────────────────────────────────────────────────────────
M = {
    1: 4,
    2: 5,
    3: 3,
    4: 2,
    5: 4,
    6: 3,
    7: 5,
    8: 4
}

# Tipos: 1 = Balada, 2 = Hit
A = {
    (1, 1): 1, (1, 2): 0,
    (2, 1): 0, (2, 2): 1,
    (3, 1): 1, (3, 2): 0,
    (4, 1): 0, (4, 2): 1,
    (5, 1): 1, (5, 2): 0,
    (6, 1): 0, (6, 2): 1,
    (7, 1): 0, (7, 2): 0,
    (8, 1): 1, (8, 2): 1
}

# ─────────────────────────────────────────────────────────────
# Modelo
# ─────────────────────────────────────────────────────────────
model = lp.LpProblem("Asignacion_Canciones", lp.LpMinimize)

# ─────────────────────────────────────────────────────────────
# Variables de decisión
# ─────────────────────────────────────────────────────────────

#X[cp] = 1 si la cancion c pertenece a la parte p, de lo contrario, 0

X = lp.LpVariable.dicts("X",[(c, p) for c in C for p in P],cat="Binary")

# ─────────────────────────────────────────────────────────────
# Función objetivo
# ─────────────────────────────────────────────────────────────
model += 0


# ─────────────────────────────────────────────────────────────
# Restricciones
# ─────────────────────────────────────────────────────────────
# 1. Cada parte debe tener exactamente dos baladas
for p in P:
    model += lp.lpSum(X[(c, p)] * A[(c, 1)] for c in C) == 2

# 2. Parte 1 debe tener al menos tres hits
model += lp.lpSum(X[(c, 1)] * A[(c, 2)] for c in C) >= 3

# 3. Canción 5 o canción 6 debe estar en parte 1
model += X[(5, 1)] + X[(6, 1)] >= 1

# 4. Si canciones 2 y 4 están en parte 1, canción 5 debe estar en parte 2
model += X[(2, 1)] + X[(4, 1)] - 1 <= X[(5, 2)]

# 5. Cada parte debe durar entre 14 y 16 minutos
for p in P:
    model += lp.lpSum(M[c] * X[(c, p)] for c in C) >= 14
    model += lp.lpSum(M[c] * X[(c, p)] for c in C) <= 16

# 6. Cada canción debe asignarse a una sola parte
for c in C:
    model += lp.lpSum(X[(c, p)] for p in P) == 1

# ─────────────────────────────────────────────────────────────
# Resolver
# ─────────────────────────────────────────────────────────────
model.solve()

# ─────────────────────────────────────────────────────────────
# Resultados
# ─────────────────────────────────────────────────────────────
print("Estado:", lp.LpStatus[model.status])

for p in P:
    canciones = [c for c in C if lp.value(X[(c, p)]) == 1]
    duracion = sum(M[c] for c in canciones)
    baladas = sum(A[(c, 1)] for c in canciones)
    hits = sum(A[(c, 2)] for c in canciones)

    print(f"\nParte {p}:")
    print("Canciones:", canciones)
    print("Duración:", duracion)
    print("Baladas:", baladas)
    print("Hits:", hits)

Estado: Optimal

Parte 1:
Canciones: [1, 2, 6, 8]
Duración: 16
Baladas: 2
Hits: 3

Parte 2:
Canciones: [3, 4, 5, 7]
Duración: 14
Baladas: 2
Hits: 1


#Ejercicio 4

In [ ]:
import pulp as lp
from pulp import *

# 1. Conjuntos

# i = tipos de combustible
combustibles = [1, 2, 3]          # 1=Súper, 2=Regular, 3=Sin Plomo
nombres_comb = {1: "Súper", 2: "Regular", 3: "Sin Plomo"}

# J = compartimientos del camión
compartimientos = [1, 2, 3, 4, 5]

# 2. Parámetros

# CJ: Capacidad de cada compartimiento (galones)
C = {1: 2700, 2: 2800, 3: 1100, 4: 1800, 5: 3400}

# Di: Demanda de cada combustible (galones)
D = {1: 2900, 2: 4000, 3: 4900}

# Pi: Penalidad por galón faltante ($/galón)
P = {1: 10, 2: 8, 3: 6}

# Si: Escasez máxima permitida por combustible (galones)
S = {1: 500, 2: 500, 3: 500}

# 3. Modelo

modelo_sunco = lp.LpProblem("Sunco_Oil_Camion", sense=lp.LpMinimize)

# 4. Variables de decisión

#1 si el compartimiento J se usa para el combustible i
X = lp.LpVariable.dicts("X", (combustibles, compartimientos), cat="Binary")

# Fi >= 0: galones de escasez del combustible i
F = lp.LpVariable.dicts("F", combustibles, lowBound=0, cat="Continuous")

# 5. Función objetivo

modelo_sunco += lp.lpSum(P[i] * F[i] for i in combustibles), "Costo_Total_Escasez"

# 6. Restricciones

# R1. Cada compartimiento J se asigna máximo a un tipo de combustible

for J in compartimientos:
    modelo_sunco += (
        lp.lpSum(X[i][J] for i in combustibles) <= 1
    ), f"R1_comp_{J}"

# R2. Balance oferta–demanda con escasez


for i in combustibles:
    modelo_sunco += (
        lp.lpSum(X[i][J] * C[J] for J in compartimientos) + F[i] >= D[i]
    ), f"R2_balance_{nombres_comb[i]}"

# R3. Escasez no puede superar el máximo permitido
# Fi <= Si  ∀i

for i in combustibles:
    modelo_sunco += F[i] <= S[i], f"R3_escasez_max_{nombres_comb[i]}"

# R4. No negatividad
# Fi >= 0  ∀i

# R5. Binariedad


# 7. Resolver

modelo_sunco.solve(PULP_CBC_CMD(msg=0))

# 8. Resultados

print("=" * 60)
print("EJERCICIO 4 — Sunco Oil: Carga del Camión")
print("=" * 60)
print(f"Estado : {lp.LpStatus[modelo_sunco.status]}")
print(f"Costo mínimo total por escasez: ${lp.value(modelo_sunco.objective):,.0f}")

print()
print("--- Asignación de compartimientos ---")
for J in compartimientos:
    asignado = None
    for i in combustibles:
        if X[i][J].value() > 0.5:
            asignado = nombres_comb[i]
    estado = asignado if asignado else "Sin asignar"
    print(f"  Compartimiento J={J} ({C[J]:,} gal): {estado}")

print()
print("--- Escasez por combustible ---")
for i in combustibles:
    cargado = sum(C[J] * X[i][J].value() for J in compartimientos)
    fi      = F[i].value()
    costo_i = P[i] * fi
    print(f"  {nombres_comb[i]:10}: Demanda={D[i]:,} | Cargado={cargado:,.0f} | "
          f"Escasez={fi:,.0f} gal | Penalidad=${costo_i:,.0f}")

EJERCICIO 4 — Sunco Oil: Carga del Camión
Estado : Optimal
Costo mínimo total por escasez: $2,600

--- Asignación de compartimientos ---
  Compartimiento J=1 (2,700 gal): Regular
  Compartimiento J=2 (2,800 gal): Súper
  Compartimiento J=3 (1,100 gal): Regular
  Compartimiento J=4 (1,800 gal): Sin Plomo
  Compartimiento J=5 (3,400 gal): Sin Plomo

--- Escasez por combustible ---
  Súper     : Demanda=2,900 | Cargado=2,800 | Escasez=100 gal | Penalidad=$1,000
  Regular   : Demanda=4,000 | Cargado=3,800 | Escasez=200 gal | Penalidad=$1,600
  Sin Plomo : Demanda=4,900 | Cargado=5,200 | Escasez=0 gal | Penalidad=$0
